# SWaT — Part 2: Evaluation & Visualisation
**Mirrors `swat_pipeline_final.ipynb` exactly:**
- `DATA_PATH = "final.csv"`
- Same `DEAD_COLS`
- Same temporal features: `_rmean{w}`, `_rstd{w}`, `_zscore50`, `_roc50`, `d_{s}_dt`
- No CUSUM, no `_dev` features

Run after `swat_pipeline_final.ipynb` (models must already be saved).


## 0 · Setup

In [ ]:
import warnings, json as _json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import joblib

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

MODELS_DIR = Path("saved_models")
PLOTS_DIR  = Path("plots"); PLOTS_DIR.mkdir(exist_ok=True)
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ATTACK_NAMES = {
    0:"Normal Operation",       1:"Tank Overflow Attack",
    2:"Chemical Depletion Attack", 3:"Membrane Damage Attack",
    4:"pH Manipulation Attack", 5:"Slow Ramp Attack",
    6:"Reconnaissance Scan",    7:"Denial of Service",
    8:"Replay Attack",          9:"Valve Manipulation Attack",
}
NUM_CLASSES = 10
ID_REMAP    = {0:0,8:1,9:2,10:3,11:4,12:5,13:6,14:7,15:8,16:9}
PALETTE     = ['#1565C0','#E53935','#2E7D32','#F57F17','#6A1B9A',
               '#00838F','#AD1457','#37474F','#558B2F','#4E342E']
SHORT_FLAT  = [f"[{i}] {ATTACK_NAMES[i][:16]}" for i in range(10)]
MC_LABELS   = [ATTACK_NAMES[i] for i in range(NUM_CLASSES)]
print(f"Device: {DEVICE}")


In [ ]:
# Load saved models
scaler   = joblib.load(MODELS_DIR/"scaler.joblib")
rf_bin   = joblib.load(MODELS_DIR/"rf_binary.joblib")
xgb_bin  = joblib.load(MODELS_DIR/"xgb_binary.joblib")
svm_bin  = joblib.load(MODELS_DIR/"svm_binary.joblib")
rf_mc    = joblib.load(MODELS_DIR/"rf_multiclass.joblib")
xgb_mc   = joblib.load(MODELS_DIR/"xgb_multiclass.joblib")
print("All models loaded OK")


## 1 · Rebuild Feature Matrix (exact mirror of final pipeline)

In [ ]:
# ── Same DATA_PATH as final pipeline ─────────────────────────────────────────
DATA_PATH = "final.csv"

df_raw = pd.read_csv(DATA_PATH)
df_raw['ATTACK_ID']   = df_raw['ATTACK_ID'].map(ID_REMAP)
df_raw['ATTACK_NAME'] = df_raw['ATTACK_ID'].map(ATTACK_NAMES)
df_raw['ts']          = pd.to_datetime(df_raw['Timestamp'])
df_raw = df_raw.sort_values(['run_id','elapsed_seconds']).reset_index(drop=True)
print(f"Loaded: {df_raw.shape}")


In [ ]:
DEAD_COLS = [
    # UF near-constants
    'UF_Last_Backwash', 'Turbidity_UF', 'UF_Runtime',
    # RO counters & flat sensors
    'RO_Runtime', 'RO_Fouling_Factor', 'RO_Last_Cleaning',
    'TDS_Feed', 'TDS_Permeate', 'RO_Cleaning_Active',
    # Energy counters — monotone, encode time not attacks (leakage!)
    'Energy_P101', 'Energy_P301', 'Energy_P501', 'Energy_Total',
    # Dead pump booleans — always same state across all runs
    'P_102', 'P_202', 'P_204', 'P_206',
    'P_301', 'P_401', 'P_404', 'P_502', 'P_603',
    # Alarm/flag always firing
    'High_Fouling_Alarm',
    # Identical to High_Level_Alarm
    'System_Run',
    # Metadata
    'Timestamp', 'ATTACK_NAME', 'MITRE_ID', 'ts',
]
DEAD_COLS = [c for c in DEAD_COLS if c in df_raw.columns]
df = df_raw.drop(columns=DEAD_COLS).copy()
print(f"Dropped {len(DEAD_COLS)} columns: {df_raw.shape[1]} -> {df.shape[1]}")


In [ ]:
# ── Leakage removal (same as final) ──────────────────────────────────────────
def remove_leakage(df, window_s=4.0):
    parts = []
    for rid, grp in df.groupby('run_id'):
        grp = grp.sort_values('elapsed_seconds').reset_index(drop=True)
        t_changes = grp['ATTACK_ID'].diff().abs() > 0
        t_times   = grp.loc[t_changes,'elapsed_seconds'].values
        mask = pd.Series(False, index=grp.index)
        for t in t_times:
            mask |= (grp['elapsed_seconds']-t).abs() <= window_s
        parts.append(grp[~mask])
    return pd.concat(parts, ignore_index=True)

df = remove_leakage(df)
print(f"After leakage removal: {len(df):,} rows")


In [ ]:
# ── Tank normalisation (same as final) ───────────────────────────────────────
TANK_COLS = [c for c in ['Acid_Tank_Level','Chlorine_Tank_Level',
                          'Coagulant_Tank_Level','Bisulfate_Tank_Level']
             if c in df.columns]
for col in TANK_COLS:
    rs = df[df.ATTACK_ID==0].groupby('run_id')[col].first().rename('_s')
    df = df.join(rs, on='run_id')
    df[f'{col}_pct'] = ((df['_s']-df[col])/df['_s'].clip(lower=1)).clip(0,1)
    df.drop(columns=['_s'], inplace=True)

# Bool → int
bool_cols = [c for c in df.columns if df[c].dtype==bool]
if bool_cols:
    df[bool_cols] = df[bool_cols].astype(np.int8)
print("Tank normalisation + bool conversion done")


In [ ]:
WINDOW_SIZES     = [10, 25, 50]
TEMPORAL_SENSORS = [c for c in
    ['LIT_101','LIT_301','LIT_401','AIT_202','FIT_101','FIT_401',
     'Acid_Tank_Level','Chlorine_Residual','DPIT_301']
    if c in df.columns]
print("Temporal sensors:", TEMPORAL_SENSORS)

def add_temporal_features(df, sensors, windows, run_col='run_id', time_col='elapsed_seconds'):
    parts = []
    for rid, grp in df.groupby(run_col):
        grp = grp.sort_values(time_col).copy().reset_index(drop=True)
        dt  = grp[time_col].diff().fillna(0.2).clip(lower=0.01)
        for s in sensors:
            if s not in grp.columns: continue
            col_v = grp[s].astype(float)
            # Derivative
            grp[f'd_{s}_dt'] = col_v.diff().fillna(0) / dt
            # Rolling stats — min_periods=1 prevents NaN
            for w in windows:
                grp[f'{s}_rmean{w}'] = col_v.rolling(w, min_periods=1).mean()
                grp[f'{s}_rstd{w}']  = col_v.rolling(w, min_periods=1).std().fillna(0)
            # Z-score over 50-sample window
            roll50 = col_v.rolling(50, min_periods=10)
            grp[f'{s}_zscore50'] = ((col_v - roll50.mean()) / roll50.std().clip(lower=0.001)).fillna(0)
            # Rate-of-change over 50 samples (captures slow ramp)
            grp[f'{s}_roc50'] = (col_v - col_v.shift(50).fillna(method='bfill')) / 50
        # Per-run NaN sweep
        new_c = [c for c in grp.columns
                 if any(c.startswith(p) for p in [f'd_', *(f'{s}_' for s in sensors)])
                 and c not in df.columns]
        for c in new_c:
            if grp[c].isna().any():
                grp[c] = grp[c].ffill().bfill().fillna(0)
        parts.append(grp)
    return pd.concat(parts, ignore_index=True)

print("Computing temporal features (may take ~30s)...")
df = add_temporal_features(df, TEMPORAL_SENSORS, WINDOW_SIZES)
new_feats = [c for c in df.columns if any(x in c for x in ['_rmean','_rstd','_zscore50','_roc50','d_'])]
print(f"Added {len(new_feats)} temporal features  |  Total columns: {df.shape[1]}")


In [ ]:
NON_FEAT = {'ATTACK_ID','run_id','elapsed_seconds'}
FEATURE_COLS = [c for c in df.columns
                if c not in NON_FEAT
                and df[c].dtype in (np.float64,np.float32,np.int64,
                                    np.int32,np.int16,np.int8,float,int)]
print(f"Feature columns: {len(FEATURE_COLS)}")

X_raw = df[FEATURE_COLS].copy()
X_raw['_run'] = df['run_id'].values
X_filled = (X_raw.groupby('_run', group_keys=False)
            .apply(lambda g: g.ffill().bfill())
            .fillna(0)).drop(columns=['_run'])
inf_mask = np.isinf(X_filled.values)
if inf_mask.any():
    for i, c in enumerate(X_filled.columns):
        ci = np.isinf(X_filled[c].values)
        if ci.any():
            X_filled.loc[ci, c] = np.nanmedian(X_filled[c].values[~ci])

X = X_filled.values.astype(np.float32)
y = df['ATTACK_ID'].values.astype(np.int64)
y_binary = (y > 0).astype(np.int64)
assert np.isnan(X).sum() == 0, "NaN in feature matrix!"
print(f"X: {X.shape}  |  NaN: {np.isnan(X).sum()}  AUDIT PASSED")


In [ ]:
# Split + scale (same as final)
X_tr, X_te, y_tr, y_te, yb_tr, yb_te = train_test_split(
    X, y, y_binary, test_size=0.20, random_state=42, stratify=y)
X_tr_sc = scaler.transform(X_tr)
X_te_sc = scaler.transform(X_te)
assert np.isnan(X_te_sc).sum() == 0
print(f"Train: {X_tr_sc.shape}  Test: {X_te_sc.shape}")


## 2 · Class Distribution

In [ ]:
counts = df.groupby('ATTACK_ID').size()
fig, axes = plt.subplots(1,2,figsize=(14,5))
short = [f"[{i}] {ATTACK_NAMES[i][:16]}" for i in counts.index]
axes[0].barh(short[::-1], counts.values[::-1], color=[PALETTE[i] for i in counts.index[::-1]])
axes[0].set_xlabel("Row count"); axes[0].set_title("Rows per Class", fontweight='bold')
for i,v in enumerate(counts.values[::-1]):
    axes[0].text(v+200, i, f"{v:,}", va='center', fontsize=8)
axes[1].pie(counts.values, labels=[f"[{i}]" for i in counts.index],
            colors=PALETTE, autopct='%1.1f%%', startangle=140, textprops={'fontsize':8})
patches = [mpatches.Patch(color=PALETTE[i], label=f"[{i}] {ATTACK_NAMES[i]}") for i in range(10)]
axes[1].legend(handles=patches, bbox_to_anchor=(1.05,1), fontsize=7)
axes[1].set_title("Distribution", fontweight='bold')
plt.suptitle(f"SWaT — {len(df):,} rows · 4 runs", fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"01_class_dist.png", dpi=150, bbox_inches='tight'); plt.show()


## 3 · Sensor Time-Series per Attack

In [ ]:
KEY_SENSORS = ['LIT_101','LIT_301','LIT_401','AIT_202',
               'Acid_Tank_Level','Chlorine_Residual','FIT_401','MV_101']
ATTACK_PLOT = {1:"Tank Overflow",2:"Chem Deplet",4:"pH Manip",5:"Slow Ramp",9:"Valve Manip"}

r1 = df[df['run_id']==1].sort_values('elapsed_seconds').reset_index(drop=True)
time_min = r1['elapsed_seconds'].values/60

fig, axes = plt.subplots(len(KEY_SENSORS),1,figsize=(16,22),sharex=False)
fig.subplots_adjust(hspace=0.55)
for ax, sensor in zip(axes, KEY_SENSORS):
    if sensor not in r1.columns: continue
    for _, seg in r1.groupby((r1['ATTACK_ID']!=r1['ATTACK_ID'].shift()).cumsum()):
        aid = seg['ATTACK_ID'].iloc[0]
        if aid==0: continue
        ax.axvspan(seg['elapsed_seconds'].iloc[0]/60, seg['elapsed_seconds'].iloc[-1]/60,
                   alpha=0.18, color=PALETTE[aid])
    ax.plot(time_min, r1[sensor].values.astype(float), color='#212121', lw=0.7, alpha=0.85)
    ax.set_ylabel(sensor, fontsize=8, fontweight='bold')
    ax.set_xlabel("Time (min)", fontsize=7); ax.grid(alpha=0.25)
    for aid,name in ATTACK_PLOT.items():
        seg=r1[r1['ATTACK_ID']==aid]
        if not len(seg): continue
        ax.axvline(seg['elapsed_seconds'].iloc[0]/60,color=PALETTE[aid],lw=1.2,ls='--',alpha=0.7)
        ax.text(seg['elapsed_seconds'].mean()/60, ax.get_ylim()[1]*0.92,
                name[:8],fontsize=6,color=PALETTE[aid],ha='center',fontweight='bold')
plt.suptitle("Sensor Time-Series — Run 1  (shaded = attack windows)",
             fontweight='bold',fontsize=13,y=1.002)
plt.savefig(PLOTS_DIR/"02_sensor_timeseries.png",dpi=150,bbox_inches='tight'); plt.show()


## 4 · Feature Correlation Heatmap (Top 30)

In [ ]:
top30 = (pd.DataFrame(X_tr_sc, columns=FEATURE_COLS)
         .corrwith(pd.Series(yb_tr.astype(float)))
         .abs().sort_values(ascending=False).head(30).index.tolist())
corr_mat = pd.DataFrame(X_te_sc, columns=FEATURE_COLS)[top30].corr()
mask = np.triu(np.ones_like(corr_mat,dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(14,12))
sns.heatmap(corr_mat, mask=mask, cmap='coolwarm', center=0, annot=False,
            linewidths=0.3, ax=ax, cbar_kws={"shrink":0.8,"label":"Pearson r"})
ax.set_title("Feature Correlation — Top 30 Features", fontweight='bold', fontsize=12)
ax.tick_params(axis='x',rotation=45,labelsize=7)
ax.tick_params(axis='y',rotation=0,labelsize=7)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"03_corr_heatmap.png",dpi=150,bbox_inches='tight'); plt.show()


## 5 · SMOTE Class Balancing

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=3, sampling_strategy={2:4000,3:4000,7:4000})
X_bal, y_bal = smote.fit_resample(X_tr_sc, y_tr)
y_bin_bal    = (y_bal>0).astype(np.int64)
before=Counter(y_tr); after=Counter(y_bal)

fig, ax = plt.subplots(figsize=(13,5))
x=np.arange(NUM_CLASSES); w=0.38
ax.bar(x-w/2,[before.get(i,0) for i in range(NUM_CLASSES)],w,
       label='Before',color='#90CAF9',edgecolor='#1565C0',lw=0.8)
ax.bar(x+w/2,[after.get(i,0) for i in range(NUM_CLASSES)],w,
       label='After',color='#EF9A9A',edgecolor='#C62828',lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"[{i}] {ATTACK_NAMES[i][:14]}" for i in range(NUM_CLASSES)],
                   rotation=30,ha='right',fontsize=8)
ax.set_title("SMOTE — Training Set Class Counts",fontweight='bold')
ax.set_ylabel("Samples"); ax.legend(); ax.grid(axis='y',alpha=0.3)
for xi,(b,a) in enumerate(zip([before.get(i,0) for i in range(NUM_CLASSES)],
                               [after.get(i,0) for i in range(NUM_CLASSES)])):
    if a!=b: ax.annotate(f"+{a-b:,}",xy=(xi+w/2,a),ha='center',va='bottom',fontsize=7,color='#C62828')
plt.tight_layout()
plt.savefig(PLOTS_DIR/"04_smote.png",dpi=150,bbox_inches='tight'); plt.show()
print(f"After SMOTE: {len(X_bal):,} training samples")


## 6 · MLP Training Curves

In [ ]:
class SWaTMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def make_loader(Xtr,ytr,Xte,yte,batch=1024):
    tr=DataLoader(TensorDataset(torch.FloatTensor(Xtr),torch.LongTensor(ytr)),
                  batch_size=batch,shuffle=True,num_workers=0)
    te=DataLoader(TensorDataset(torch.FloatTensor(Xte),torch.LongTensor(yte)),
                  batch_size=batch,shuffle=False,num_workers=0)
    return tr,te

def train_epoch(model,loader,crit,opt,device):
    model.train(); tl,tc,tn=0.,0,0
    for Xb,yb in loader:
        Xb,yb=Xb.to(device),yb.to(device); opt.zero_grad()
        out=model(Xb); loss=crit(out,yb); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step()
        tl+=loss.item()*len(yb); tc+=(out.argmax(1)==yb).sum().item(); tn+=len(yb)
    return tl/tn, tc/tn

def eval_epoch(model,loader,crit,device):
    model.eval(); vl,vc,vn=0.,0,0; pa,ta=[],[]
    with torch.no_grad():
        for Xb,yb in loader:
            Xb,yb=Xb.to(device),yb.to(device)
            out=model(Xb); vl+=crit(out,yb).item()*len(yb)
            p=out.argmax(1); vc+=(p==yb).sum().item(); vn+=len(yb)
            pa.extend(p.cpu().numpy()); ta.extend(yb.cpu().numpy())
    from sklearn.metrics import f1_score as _f1
    return vl/vn, vc/vn, _f1(ta,pa,average='macro',zero_division=0), np.array(pa)

cw_mc  = dict(zip(np.unique(y_bal).tolist(),
               compute_class_weight('balanced',classes=np.unique(y_bal),y=y_bal).tolist()))
INPUT_DIM = X_bal.shape[1]
mlp_mc = SWaTMLP(INPUT_DIM,[512,256,128,64],NUM_CLASSES).to(DEVICE)
tr_mc,te_mc = make_loader(X_bal,y_bal,X_te_sc,y_te)
cw_t = torch.FloatTensor([cw_mc.get(i,1.) for i in range(NUM_CLASSES)]).to(DEVICE)
crit_mc = nn.CrossEntropyLoss(weight=cw_t)
opt_mc  = torch.optim.Adam(mlp_mc.parameters(),lr=1e-3,weight_decay=1e-5)
sched_mc = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_mc,patience=5,factor=0.5)

EPOCHS=50; best_f1,best_preds=0.,None
hist={'tr_loss':[],'va_loss':[],'tr_acc':[],'va_acc':[],'va_f1':[]}
print(f"Training MLP ({EPOCHS} epochs on {DEVICE})...")
for ep in range(1,EPOCHS+1):
    tl,ta=train_epoch(mlp_mc,tr_mc,crit_mc,opt_mc,DEVICE)
    vl,va,vf,preds=eval_epoch(mlp_mc,te_mc,crit_mc,DEVICE)
    sched_mc.step(vl)
    for k,v in zip(['tr_loss','va_loss','tr_acc','va_acc','va_f1'],[tl,vl,ta,va,vf]):
        hist[k].append(v)
    if vf>best_f1: best_f1,best_preds=vf,preds.copy(); torch.save(mlp_mc.state_dict(),MODELS_DIR/"mlp_multiclass_best.pt")
    if ep%10==0 or ep==1: print(f"  Ep{ep:3d}  TrAcc={ta:.4f}  VaAcc={va:.4f}  VaF1={vf:.4f}")
print(f"Best F1: {best_f1:.4f}")


In [ ]:
ep_x=range(1,EPOCHS+1)
fig,axes=plt.subplots(2,3,figsize=(16,9))
for ax,(tk,vk,title) in zip(axes.flat[:3],[
    ('tr_loss','va_loss','Loss'),('tr_acc','va_acc','Accuracy'),('va_f1',None,'Macro F1')]):
    ax.plot(ep_x,hist[tk],color='#1565C0',lw=1.8,label='Train')
    if vk: ax.plot(ep_x,hist[vk],color='#E53935',lw=1.8,ls='--',label='Val')
    if title=='Macro F1': ax.axhline(best_f1,color='grey',ls=':',label=f'Best={best_f1:.4f}')
    ax.set_title(f"MLP {title}",fontweight='bold'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
from scipy.ndimage import uniform_filter1d
ax_sm=axes[1,0]
sm=uniform_filter1d(hist['va_f1'],size=5)
ax_sm.plot(ep_x,hist['va_f1'],alpha=0.3,color='green',lw=1)
ax_sm.plot(ep_x,sm,color='green',lw=2.2,label='Smoothed F1')
ax_sm.axhline(best_f1,color='grey',ls=':',label=f'Best={best_f1:.4f}')
ax_sm.set_title("Smoothed Val Macro F1",fontweight='bold'); ax_sm.legend(fontsize=8); ax_sm.grid(alpha=0.3)
ax_gap=axes[1,1]
gap=np.array(hist['va_loss'])-np.array(hist['tr_loss'])
ax_gap.fill_between(ep_x,0,gap,where=gap>0,alpha=0.4,color='red',label='Overfit')
ax_gap.fill_between(ep_x,0,gap,where=gap<0,alpha=0.4,color='blue',label='Underfit')
ax_gap.plot(ep_x,gap,color='black',lw=1); ax_gap.axhline(0,color='grey',lw=0.8)
ax_gap.set_title("Val-Train Loss Gap",fontweight='bold'); ax_gap.legend(fontsize=8); ax_gap.grid(alpha=0.3)
axes[1,2].axis('off')
plt.suptitle("MLP Training Diagnostics",fontweight='bold',fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"05_training_curves.png",dpi=150,bbox_inches='tight'); plt.show()


## 7 · Confusion Matrices — All Models

In [ ]:
def plot_cm(y_true, y_pred, title, fname, cmap='Blues'):
    cm=confusion_matrix(y_true,y_pred,normalize='true')
    fig,axes=plt.subplots(1,2,figsize=(20,8),gridspec_kw={'width_ratios':[3,1]})
    sns.heatmap(cm,annot=True,fmt='.2f',cmap=cmap,
                xticklabels=SHORT_FLAT,yticklabels=SHORT_FLAT,
                ax=axes[0],linewidths=0.4,annot_kws={'size':8})
    axes[0].set_title(title,fontweight='bold',fontsize=12)
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    axes[0].tick_params(axis='x',rotation=45,labelsize=7.5)
    axes[0].tick_params(axis='y',rotation=0,labelsize=7.5)
    diag=cm.diagonal()
    axes[1].barh(SHORT_FLAT[::-1],diag[::-1],color=[PALETTE[i] for i in range(10)][::-1])
    axes[1].set_xlim(0,1.1); axes[1].set_title("Per-class
Recall",fontweight='bold')
    for i,v in enumerate(diag[::-1]):
        axes[1].text(v+0.01,i,f"{v:.2f}",va='center',fontsize=8)
    axes[1].grid(axis='x',alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR/fname,dpi=150,bbox_inches='tight'); plt.show()
    print(f"  {title}: Accuracy={accuracy_score(y_true,y_pred):.4f}  Macro F1={f1_score(y_true,y_pred,average='macro',zero_division=0):.4f}")

rf_preds  = rf_mc.predict(X_te_sc)
xgb_preds = xgb_mc.predict(X_te_sc)
plot_cm(y_te,best_preds,"MLP Multi-class CM","06_cm_mlp.png",cmap='Greens')
plot_cm(y_te,rf_preds,  "Random Forest MC CM","06_cm_rf.png",cmap='Blues')
plot_cm(y_te,xgb_preds, "XGBoost MC CM","06_cm_xgb.png",cmap='Oranges')

fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,(mdl,name,cm_) in zip(axes,[(rf_bin,"RF Binary",'Blues'),
                                    (xgb_bin,"XGB Binary",'Oranges'),
                                    (svm_bin,"SVM Binary",'Purples')]):
    p=mdl.predict(X_te_sc)
    cm=confusion_matrix(yb_te,p,normalize='true')
    sns.heatmap(cm,annot=True,fmt='.3f',cmap=cm_,
                xticklabels=['Normal','Attack'],yticklabels=['Normal','Attack'],
                ax=ax,linewidths=1,annot_kws={'size':13})
    ax.set_title(f"{name}  F1={f1_score(yb_te,p,average='macro',zero_division=0):.4f}",fontweight='bold')
plt.suptitle("Binary Detection — Confusion Matrices",fontweight='bold',fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"06_cm_binary.png",dpi=150,bbox_inches='tight'); plt.show()


## 8 · ROC Curves

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,(mdl,name,col) in zip(axes,[(rf_bin,"Random Forest",'#1565C0'),
                                    (xgb_bin,"XGBoost",'#2E7D32'),
                                    (svm_bin,"SVM RBF",'#E65100')]):
    proba=mdl.predict_proba(X_te_sc)[:,1]
    fpr,tpr,thr=roc_curve(yb_te,proba); auc=roc_auc_score(yb_te,proba)
    ax.plot(fpr,tpr,lw=2.5,color=col,label=f"AUC = {auc:.4f}")
    ax.plot([0,1],[0,1],'--',color='grey',lw=1)
    j=np.argmax(tpr-fpr); ax.scatter(fpr[j],tpr[j],s=80,color='red',zorder=5,label=f"thr={thr[j]:.3f}")
    ax.fill_between(fpr,tpr,alpha=0.07,color=col)
    ax.set_title(name,fontweight='bold'); ax.legend(fontsize=9)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.grid(alpha=0.3)
plt.suptitle("ROC Curves — Binary Detection",fontweight='bold',fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"07_roc_binary.png",dpi=150,bbox_inches='tight'); plt.show()

y_ohe=label_binarize(y_te,classes=list(range(NUM_CLASSES))); xgb_prb=xgb_mc.predict_proba(X_te_sc)
fig,axes=plt.subplots(2,5,figsize=(20,9))
for i,ax in enumerate(axes.flat):
    fpr,tpr,_=roc_curve(y_ohe[:,i],xgb_prb[:,i]); auc=roc_auc_score(y_ohe[:,i],xgb_prb[:,i])
    ax.plot(fpr,tpr,lw=2,color=PALETTE[i],label=f"AUC={auc:.3f}")
    ax.plot([0,1],[0,1],':',color='grey',lw=1); ax.fill_between(fpr,tpr,alpha=0.07,color=PALETTE[i])
    ax.set_title(f"[{i}] {ATTACK_NAMES[i][:18]}",fontsize=8,fontweight='bold',color=PALETTE[i])
    ax.legend(fontsize=8,loc='lower right'); ax.set_xlabel("FPR",fontsize=7); ax.grid(alpha=0.3)
plt.suptitle("One-vs-Rest ROC — XGBoost",fontweight='bold',fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"08_roc_mc.png",dpi=150,bbox_inches='tight'); plt.show()


## 9 · Precision-Recall Curves

In [ ]:
rf_prb=rf_mc.predict_proba(X_te_sc)
fig,axes=plt.subplots(2,5,figsize=(20,9))
for i,ax in enumerate(axes.flat):
    for prb,mname,col in [(xgb_prb,"XGB",'#E65100'),(rf_prb,"RF",'#1565C0')]:
        prec,rec,_=precision_recall_curve(y_ohe[:,i],prb[:,i])
        ap=average_precision_score(y_ohe[:,i],prb[:,i])
        ax.plot(rec,prec,lw=2,color=col,label=f"{mname} AP={ap:.3f}")
    base=y_ohe[:,i].mean()
    ax.axhline(base,color='grey',ls=':',lw=1)
    ax.set_title(f"[{i}] {ATTACK_NAMES[i][:18]}",fontsize=8,fontweight='bold',color=PALETTE[i])
    ax.legend(fontsize=7); ax.set_xlim(0,1); ax.set_ylim(0,1.05); ax.grid(alpha=0.3)
plt.suptitle("Precision-Recall Curves",fontweight='bold',fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"09_pr_curves.png",dpi=150,bbox_inches='tight'); plt.show()


## 10 · Feature Importance (RF + XGBoost)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(18,11))
for ax,imp_arr,name,cmap_ in [
    (axes[0],rf_mc.feature_importances_,"Random Forest",'Blues_r'),
    (axes[1],xgb_mc.feature_importances_,"XGBoost",'Oranges_r'),
]:
    imp=pd.Series(imp_arr,index=FEATURE_COLS).sort_values(ascending=False).head(30)
    ax.barh(imp.index[::-1],imp.values[::-1],
            color=plt.cm.get_cmap(cmap_)(np.linspace(0.3,0.9,30)))
    ax.set_title(f"Top 30 — {name}",fontweight='bold')
    ax.set_xlabel("Gini Importance"); ax.grid(axis='x',alpha=0.3)
plt.suptitle("Feature Importance",fontweight='bold',fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"10_feature_importance.png",dpi=150,bbox_inches='tight'); plt.show()


## 11 · Per-Class F1 Comparison

In [ ]:
MC_RESULTS={'Random Forest':{'y_true':y_te,'y_pred':rf_preds},
            'XGBoost':{'y_true':y_te,'y_pred':xgb_preds},
            'MLP':{'y_true':y_te,'y_pred':best_preds}}
f1_rows={}
for mname,res in MC_RESULTS.items():
    rpt=classification_report(res['y_true'],res['y_pred'],output_dict=True,zero_division=0)
    f1_rows[mname]={ATTACK_NAMES[i]:rpt.get(str(i),{}).get('f1-score',0.) for i in range(NUM_CLASSES)}
f1_df=pd.DataFrame(f1_rows)

fig,axes=plt.subplots(1,2,figsize=(18,6))
x=np.arange(NUM_CLASSES); w=0.28
for k,(mname,col) in enumerate(zip(f1_rows.keys(),['#1565C0','#E65100','#2E7D32'])):
    axes[0].bar(x+k*w-w,f1_df[mname].values,w,label=mname,color=col,alpha=0.85,edgecolor='white',lw=0.5)
axes[0].set_xticks(x); axes[0].set_xticklabels([f"[{i}]" for i in range(10)],fontsize=9)
axes[0].set_ylim(0,1.12); axes[0].grid(axis='y',alpha=0.3)
axes[0].set_title("Per-Class F1",fontweight='bold'); axes[0].legend(fontsize=8)
sns.heatmap(f1_df.T,annot=True,fmt='.3f',cmap='RdYlGn',vmin=0,vmax=1,ax=axes[1],
            xticklabels=[ATTACK_NAMES[i][:14] for i in range(NUM_CLASSES)],
            linewidths=0.5,annot_kws={'size':8})
axes[1].set_title("F1 Heatmap",fontweight='bold')
axes[1].tick_params(axis='x',rotation=45,labelsize=7.5)
plt.suptitle("Per-Class F1 Comparison",fontweight='bold',fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"11_f1_comparison.png",dpi=150,bbox_inches='tight'); plt.show()
print(f1_df.round(4).to_string())


## 12 · SHAP Summary (XGBoost)

In [ ]:
try:
    import shap
    from sklearn.model_selection import StratifiedShuffleSplit
    sss=StratifiedShuffleSplit(n_splits=1,test_size=1500,random_state=42)
    _,idx=next(sss.split(X_te_sc,y_te))
    explainer=shap.TreeExplainer(xgb_mc)
    shap_values=explainer.shap_values(X_te_sc[idx])
    mean_abs=np.abs(np.array(shap_values)).mean(axis=(0,1))
    shap_imp=pd.Series(mean_abs,index=FEATURE_COLS).sort_values(ascending=False)
    fig,ax=plt.subplots(figsize=(11,8))
    top25=shap_imp.head(25)
    ax.barh(top25.index[::-1],top25.values[::-1],
            color=plt.cm.RdYlGn(top25.values/top25.values.max()))
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("SHAP Feature Importance — XGBoost",fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR/"12_shap.png",dpi=150,bbox_inches='tight'); plt.show()
except ImportError:
    print("pip install shap  to enable this plot")


## 13 · Save Best Model Bundle

In [ ]:
scores={
    'XGBoost': f1_score(y_te,xgb_preds,average='macro',zero_division=0),
    'RandomForest': f1_score(y_te,rf_preds,average='macro',zero_division=0),
    'MLP': best_f1,
}
best_name=max(scores,key=scores.get)
print("F1 scores:")
for n,v in sorted(scores.items(),key=lambda x:-x[1]):
    print(f"  {n}: {v:.4f}{'  <-- BEST' if n==best_name else ''}")

bundle={
    'model': {'XGBoost':xgb_mc,'RandomForest':rf_mc}.get(best_name, xgb_mc),
    'scaler': scaler,
    'feature_cols': FEATURE_COLS,
    'attack_names': ATTACK_NAMES,
    'id_remap': ID_REMAP,
    'model_name': best_name,
    'macro_f1': scores[best_name],
    'num_classes': NUM_CLASSES,
    'temporal_sensors': TEMPORAL_SENSORS,
    'temporal_windows': WINDOW_SIZES,
    'dead_cols': DEAD_COLS,
    'feature_type': 'rmean_rstd_zscore50_roc50',
    'seq_len': 25,
    'training_info': {'dataset':DATA_PATH,'total_rows':int(len(df)),
                      'smote_targets':{2:4000,3:4000,7:4000},
                      'leakage_window_s':4.0}
}
joblib.dump(bundle,MODELS_DIR/"best_model_bundle.joblib",compress=3)
print(f"Saved best_model_bundle.joblib")

print("\n=== All plots ===")
for p in sorted(PLOTS_DIR.glob("*.png")):
    print(f"  {p.name:<45s}  {p.stat().st_size/1024:.0f} KB")
